In [6]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer


# === 1. Drop constant columns ===
class ConstantColumnRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.cols_to_keep = [col for col in X.columns if X[col].nunique() > 1]
        return self
    def transform(self, X):
        return X[self.cols_to_keep]
    
# === 2. NAFiller ===
class NAFiller:
    def __init__(self, num_strategy='constant', num_fill_value=999,
                 cat_strategy='constant', cat_fill_value='Missing'):
        self.num_strategy = num_strategy
        self.num_fill_value = num_fill_value
        self.cat_strategy = cat_strategy
        self.cat_fill_value = cat_fill_value
        self.num_imputer = None
        self.cat_imputer = None
        self.num_cols = []
        self.cat_cols = []

    def fit(self, X, y=None):
        # Identify numerical and categorical columns
        self.num_cols = X.select_dtypes(include=['number']).columns.tolist()
        self.cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        
        # Fit numerical imputer
        self.num_imputer = SimpleImputer(strategy=self.num_strategy, fill_value=self.num_fill_value)
        if self.num_cols:
            self.num_imputer.fit(X[self.num_cols])
        
        # Fit categorical imputer
        self.cat_imputer = SimpleImputer(strategy=self.cat_strategy, fill_value=self.cat_fill_value)
        if self.cat_cols:
            self.cat_imputer.fit(X[self.cat_cols])
        
        print("finished fitting NAFiller")

        return self

    def transform(self, X):
        X = X.copy()
        # Transform numerical columns
        if self.num_cols:
            X[self.num_cols] = self.num_imputer.transform(X[self.num_cols])
        # Transform categorical columns
        if self.cat_cols:
            X[self.cat_cols] = self.cat_imputer.transform(X[self.cat_cols])
        print("finished transforming NAFiller")
        return X

# === 3. Encoder ===

class Encoder(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=3):
        self.threshold = threshold
        self.ohe_cols = []
        self.woe_cols = []
        self.woe_maps = {}
        self.ohe_df_columns = []

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y).reset_index(drop=True)
        
        for col in X.select_dtypes(include=['object', 'category']):
            unique_vals = X[col].nunique(dropna=True)
            if unique_vals < self.threshold:
                self.ohe_cols.append(col)
            else:
                self.woe_cols.append(col)
                self.woe_maps[col] = self._compute_woe(X[col], y)

        if self.ohe_cols:
            ohe_df = pd.get_dummies(X[self.ohe_cols], dummy_na=True)
            self.ohe_df_columns = ohe_df.columns.tolist()
        print("finished fitting Encoder")
        
        return self

    def transform(self, X):
        X = X.copy()
        output = X.drop(columns=self.ohe_cols + self.woe_cols, errors='ignore')

        # Apply WOE encoding
        for col in self.woe_cols:
            woe_map = self.woe_maps[col]
            X[col] = X[col].map(woe_map).fillna(0)  # fill unknown categories with 0
            output[col] = X[col]

        # Apply One-Hot Encoding
        if self.ohe_cols:
            ohe_df = pd.get_dummies(X[self.ohe_cols], dummy_na=True)
            # Align with training columns to ensure consistency
            ohe_df = ohe_df.reindex(columns=self.ohe_df_columns, fill_value=0)
            output = pd.concat([output, ohe_df], axis=1)
        print("finished transforming Encoder")
        return output

    def _compute_woe(self, feature_col, target_col):
        df = pd.DataFrame({'feature': feature_col, 'target': target_col})
        df = df.dropna(subset=['feature'])  # Drop missing for WOE mapping
        grouped = df.groupby('feature')

        # total good and bad
        total_good = (df['target'] == 0).sum()
        total_bad = (df['target'] == 1).sum()

        woe_map = {}
        for val, group in grouped:
            good = (group['target'] == 0).sum()
            bad = (group['target'] == 1).sum()

            # Avoid division by zero
            epsilon = 1e-6
            good_ratio = good / total_good if total_good else epsilon
            bad_ratio = bad / total_bad if total_bad else epsilon
            woe = np.log((good_ratio + epsilon) / (bad_ratio + epsilon))

            woe_map[val] = woe

        return woe_map



# === 4. Feature Engineering ===
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, origin_date=pd.Timestamp('2017-11-30')):
        self.origin_date = origin_date

    def fit(self, X, y=None):
        print("finished fitting FeatureEngineer")
        return self

    def transform(self, X):
        X = X.copy()
        if 'TransactionDT' in X.columns:
            X['datetime'] = self.origin_date + pd.to_timedelta(X['TransactionDT'], unit='s')
            X['weekday'] = X['datetime'].dt.weekday
            X['month'] = X['datetime'].dt.month
            X.drop(columns=['datetime'], inplace=True)

        if 'TransactionAmt' in X.columns and 'card1' in X.columns:
            X['card1_avg_amt'] = X.groupby('card1')['TransactionAmt'].transform('mean')
        print('finished transforming FeatureEngineer')
        return X

# === 5. Correlation Filter ===
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold
        self.cols_to_keep = []
        self.corr_matrix = None

    def fit(self, X, y=None):
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > self.threshold)]
        self.cols_to_keep = [col for col in X.columns if col not in to_drop]
        self.corr_matrix = corr

        # Save outputs
        self.corr_matrix.to_csv('saved_corr_matrix.csv')
        pd.Series(self.cols_to_keep).to_csv('kept_columns.csv', index=False)
        print("finished fitting CorrelationFilter")
        return self

    def transform(self, X):
        print("finished transforming CorrelationFilter")
        return X[self.cols_to_keep]

# === 6. Ensure train/test match ===
class ColumnAligner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns = None

    def fit(self, X, y=None):
        self.columns = X.columns
        print("finished fitting ColumnAligner")
        return self

    def transform(self, X):
        for col in self.columns:
            if col not in X:
                X[col] = 0
        print("finished transforming ColumnAligner")
        return X[self.columns]
    
# === 6. scaler ===

class Scaler(BaseEstimator, TransformerMixin):
    def __init__(self, scaler=None):
        self.scaler = scaler if scaler is not None else StandardScaler()
        self.columns = []

    def fit(self, X, y=None):
        X = X.copy()
        self.columns = X.select_dtypes(include=[np.number]).columns.tolist()
        self.scaler.fit(X[self.columns])
        print("finished fitting Scaler")
        return self

    def transform(self, X):
        X = X.copy()
        X[self.columns] = self.scaler.transform(X[self.columns])
        print("finished transforming Scaler")
        return X


In [7]:
import pandas as pd
train_identity  = pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction  = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
test_identity = pd.read_csv('ieee-fraud-detection/test_identity.csv')
test_transaction = pd.read_csv('ieee-fraud-detection/test_transaction.csv')
# Normalize column names (replace '-' with '_')
train_identity.columns = train_identity.columns.str.replace('-', '_')
test_identity.columns = test_identity.columns.str.replace('-', '_')
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

In [8]:
# Usage
X = train.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y = train['isFraud']
X_test = test.drop(columns=['TransactionID'], errors='ignore')

In [ ]:
from sklearn.feature_selection import RFE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from skopt import BayesSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from skopt.space import Real, Integer

# Define pipeline
pipeline = ImbPipeline([
    ('na_filler', NAFiller()),
    ('remove_constants', ConstantColumnRemover()),
    ('feature_engineering', FeatureEngineer()),
    ('categorical_encoding', Encoder()),  # threshold is tuned
    ('undersampler', RandomUnderSampler(random_state=42)),  # strategy is tuned
    ('rfe', RFE(estimator=XGBClassifier(eval_metric='logloss', step = 20))),  # n_features is tuned
    ('classifier', XGBClassifier(eval_metric='logloss'))
])

# Define search space
search_spaces = {
    'categorical_encoding__threshold': Integer(2, 10),
    'undersampler__sampling_strategy': Real(0.05, 0.5, prior='uniform'),
    'rfe__n_features_to_select': Integer(20, 100),  # adjust based on your data

    # XGBoost hyperparams
    'classifier__n_estimators': Integer(50, 300),
    'classifier__max_depth': Integer(3, 10),
    'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'classifier__subsample': Real(0.5, 1.0, prior='uniform'),
    'classifier__colsample_bytree': Real(0.5, 1.0, prior='uniform'),
    'classifier__gamma': Real(0, 5, prior='uniform'),
    'classifier__reg_alpha': Real(0, 5, prior='uniform'),
    'classifier__reg_lambda': Real(0, 5, prior='uniform')
}


# Define cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# BayesSearchCV
opt = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_spaces,
    scoring='f1',
    cv=cv,
    n_iter=15,  # increase for better results
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# Fit to training data
opt.fit(X, y)

# Results
print("Best f1:", opt.best_score_)
print("Best parameters:")
for param, val in opt.best_params_.items():
    print(f"  {param}: {val}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
